In [20]:
import pandas as pd

df = pd.read_csv("data/clean/data_iter1.csv")

In [21]:
df.columns.tolist()

['Age',
 'Gender',
 'Country',
 'City',
 'Membership_Years',
 'Login_Frequency',
 'Session_Duration_Avg',
 'Pages_Per_Session',
 'Cart_Abandonment_Rate',
 'Wishlist_Items',
 'Total_Purchases',
 'Average_Order_Value',
 'Days_Since_Last_Purchase',
 'Discount_Usage_Rate',
 'Returns_Rate',
 'Email_Open_Rate',
 'Customer_Service_Calls',
 'Product_Reviews_Written',
 'Social_Media_Engagement_Score',
 'Mobile_App_Usage',
 'Lifetime_Value',
 'Credit_Balance',
 'Churned',
 'Signup_Quarter_Q1',
 'Signup_Quarter_Q2',
 'Signup_Quarter_Q3',
 'Signup_Quarter_Q4',
 'Payment_Method_Diversity_1.0',
 'Payment_Method_Diversity_2.0',
 'Payment_Method_Diversity_3.0',
 'Payment_Method_Diversity_4.0',
 'Payment_Method_Diversity_5.0',
 'Make_Wishlists',
 'User_Revenue',
 'Lifetime_Per_Year',
 'Gender_code',
 'monthly_income']

# Подготовка данных
Убираем данные которые не влияют на отток.  
Закодировали страну как исторический отток из этой страны, но это лишь добавило шум в датасет. Поэтому убираем страну, решаем что для компании страна не сильно влияет на отток пользователей.  
Так же убрали:  
- среднюю длительность сессии, т.к она может быть выраженна через частоту логинов и количество страниц за сессию
- количество покупок и среднюю цену заказа, т.к они и пожизненная ценность показывают схожие данные.

In [22]:
# Отбрасываем ненужные признаки
df_churn = df.drop(columns=['Signup_Quarter_Q1', 'Signup_Quarter_Q2',
                            'Signup_Quarter_Q3', 'Signup_Quarter_Q4',
                            'Payment_Method_Diversity_1.0', 'Payment_Method_Diversity_2.0',
                            'Payment_Method_Diversity_3.0', 'Payment_Method_Diversity_4.0', 'Payment_Method_Diversity_5.0',
                            'Make_Wishlists', 'User_Revenue', 'City',
                            'Session_Duration_Avg', # login_frequency + pages_per_session
                            'Total_Purchases', # lifetime value
                            'Average_Order_Value' # lifetime value
                           ])

In [23]:
df_churn = pd.get_dummies(df_churn, columns=['Gender'], drop_first=True)
df_churn = df_churn.dropna()

In [24]:
print(f"Кол-во ушедших клиентов в датасете: {(df_churn['Churned'] == 1).sum()}, а оставшихся: {(df_churn['Churned'] == 0).sum()}")

Кол-во ушедших клиентов в датасете: 14450, а оставшихся: 35550


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder

X = df_churn.drop('Churned', axis = 1)
y = df_churn['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=7175, stratify=y)


# # Закодируем страну как исторический отток, % людей которые уходят из страны.
# Не имеет эффекта, просто добавляет шум.
# encoder = TargetEncoder()
# X_train['Country_Encoded'] = encoder.fit_transform(X_train[['Country']], y_train).ravel()
# X_test['Country_Encoded'] = encoder.transform(X_test[['Country']]).ravel()


X_train.drop(columns=['Country'], inplace=True)
X_test.drop(columns=['Country'], inplace=True)

# Cтандартизация данных

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [27]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Модель Логистической регрессии

In [28]:
model_LR = LogisticRegression(max_iter=30000, solver='lbfgs')
model_LR.fit(X_train, y_train)

LogisticRegression(max_iter=30000)

In [29]:
y_pred_test_LR = model_LR.predict(X_test)
y_pred_train_LR = model_LR.predict(X_train)

In [30]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [31]:
def print_stats(pred_test, pred_train):
    print(f"TEST:  Accuracy — {accuracy_score(y_test, pred_test)}")
    print(f"TRAIN: Accuracy — {accuracy_score(y_train, pred_train)}")
    print(f"TEST:  Precision — {precision_score(y_test, pred_test)}")
    print(f"TRAIN: Precision — {precision_score(y_train, pred_train)}")
    print(f"TEST:  Recall — {recall_score(y_test, pred_test)}")
    print(f"TRAIN: Recall — {recall_score(y_train, pred_train)}")
    print(f"TEST:  F1 — {f1_score(y_test, pred_test)}")
    print(f"TRAIN: F1 — {f1_score(y_train, pred_train)}")

print_stats(y_pred_test_LR, y_pred_train_LR)

TEST:  Accuracy — 0.7728
TRAIN: Accuracy — 0.7757866666666666
TEST:  Precision — 0.682053697597739
TRAIN: Precision — 0.6890860968394831
TEST:  Recall — 0.40077497924162747
TRAIN: Recall — 0.40841561317707853
TEST:  F1 — 0.504881450488145
TRAIN: F1 — 0.5128621089223638


# Модель КNN

In [32]:
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier(21)

model_knn.fit(X_train, y_train)

y_pred_train_KNN = model_knn.predict(X_train)
y_pred_test_KNN  = model_knn.predict(X_test)



In [33]:
print_stats(y_pred_test_KNN, y_pred_train_KNN)

TEST:  Accuracy — 0.81096
TRAIN: Accuracy — 0.8267733333333334
TEST:  Precision — 0.8426535087719298
TRAIN: Precision — 0.8754540736896731
TEST:  Recall — 0.4254082479933573
TRAIN: Recall — 0.4670111654516933
TEST:  F1 — 0.5653853227883023
TRAIN: F1 — 0.6090985678180286


In [34]:
p_test = model_knn.predict_proba(X_test)[:, 1]
p_train = model_knn.predict_proba(X_train)[:, 1]
thresh = 0.3
y_pred_test_KNN_higher_thresh = (p_test >= thresh).astype(int)
y_pred_train_KNN_higher_thresh = (p_train >= thresh).astype(int)
print_stats(y_pred_test_KNN_higher_thresh, y_pred_train_KNN_higher_thresh)

TEST:  Accuracy — 0.81416
TRAIN: Accuracy — 0.84016
TEST:  Precision — 0.6699157007376185
TRAIN: Precision — 0.7093093612239606
TEST:  Recall — 0.7038472183780792
TRAIN: Recall — 0.7572206330165175
TEST:  F1 — 0.686462410581725
TRAIN: F1 — 0.7324823707935374


In [35]:
p_test = model_knn.predict_proba(X_test)[:, 1]
p_train = model_knn.predict_proba(X_train)[:, 1]
thresh = 0.91
y_pred_test_KNN_higher_thresh = (p_test >= thresh).astype(int)
y_pred_train_KNN_higher_thresh = (p_train >= thresh).astype(int)
print_stats(y_pred_test_KNN_higher_thresh, y_pred_train_KNN_higher_thresh)

TEST:  Accuracy — 0.71896
TRAIN: Accuracy — 0.7190666666666666
TEST:  Precision — 0.9545454545454546
TRAIN: Precision — 0.9839743589743589
TEST:  Recall — 0.029061721561029614
TRAIN: Recall — 0.028328873304420042
TEST:  F1 — 0.05640612409347301
TRAIN: F1 — 0.05507220378509283


# Модель деревья решений

In [36]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
model_dtree = DecisionTreeClassifier(max_depth=6, random_state=7175)
model_dtree.fit(X_train, y_train)

y_pred_train_dtree = model_dtree.predict(X_train)
y_pred_test_dtree  = model_dtree.predict(X_test)

In [37]:
print_stats(y_pred_test_dtree, y_pred_train_dtree)

TEST:  Accuracy — 0.874
TRAIN: Accuracy — 0.8886933333333333
TEST:  Precision — 0.8468345813478557
TRAIN: Precision — 0.8813966800228964
TEST:  Recall — 0.6886244118461112
TRAIN: Recall — 0.7104364676571007
TEST:  F1 — 0.7595786902762937
TRAIN: F1 — 0.7867361536889433


90% уверенность, 33% людей пропуск.

In [38]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(100, 100))
# plot_tree(model_dtree, filled=True)
# plt.show()

"Снайперская" модель, очень высокая точность, но "не видит" 60% людей которые уйдут. Здесь можно будет использовать дорогостоящие методы методы воздействия:

In [39]:
p_test = model_dtree.predict_proba(X_test)[:, 1]
p_train = model_dtree.predict_proba(X_train)[:, 1]
thresh = 0.96
y_pred_test_dtree_higher_thresh = (p_test >= thresh).astype(int)
y_pred_train_dtree_higher_thresh = (p_train >= thresh).astype(int)
print_stats(y_pred_test_dtree_higher_thresh, y_pred_train_dtree_higher_thresh)

TEST:  Accuracy — 0.81904
TRAIN: Accuracy — 0.8191466666666667
TEST:  Precision — 0.9970566593083149
TRAIN: Precision — 1.0
TEST:  Recall — 0.37503459728757266
TRAIN: Recall — 0.3741810464150595
TEST:  F1 — 0.5450522928399034
TRAIN: F1 — 0.5445876980929358


"Общая" модель, пропускает только 20% людей которые уйдут, но имеет точность только 76%. Здесь можно будет применять менее дорогостоящие методы воздействия:

In [41]:
p_test = model_dtree.predict_proba(X_test)[:, 1]
p_train = model_dtree.predict_proba(X_train)[:, 1]
thresh = 0.3
y_pred_test_dtree_higher_thresh = (p_test >= thresh).astype(int)
y_pred_train_dtree_higher_thresh = (p_train >= thresh).astype(int)
print_stats(y_pred_test_dtree_higher_thresh, y_pred_train_dtree_higher_thresh)

TEST:  Accuracy — 0.86688
TRAIN: Accuracy — 0.88144
TEST:  Precision — 0.7867019711679906
TRAIN: Precision — 0.8133150308853809
TEST:  Recall — 0.7401051757542209
TRAIN: Recall — 0.765433237980991
TEST:  F1 — 0.7626925270964061
TRAIN: F1 — 0.7886480319452367


# Сравнение моделей
Логистическая регрессия показала параметры хуже чем две другие модели. KNN модель дала результаты с которыми можно работать, но recall был слишком низким. Используя свой порог можно было получить результаты пригодные для бизнесс решений(0.70 recall, 0.69 precision).

Модель деревья решений показала самые лучшие результаты с очень хорошими показателями.
В бизнессе нужно будет применять именно её. С помощью порога можно будет разделить методы вмешательства: 
- более дорогостоящие, при высокой уверенности модели и при оправдывающем это действие ltv клиента.
- менее дорогостоящие, при менее высокой уверенности модели.